In [12]:
import sys
sys.path.append("..")

import torch
import torch.optim as optim
from transformers import T5ForConditionalGeneration, T5Tokenizer
import time
from typing import List, Tuple
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
# from .stylistic_vector.style_metrics import style_loss
from stylistic_vector.style_metrics import style_loss


FileNotFoundError: [Errno 2] No such file or directory: 'style_mean.npy'

In [ ]:
# Cell 2: Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

model_name = "google/flan-t5-base"
print(f"Loading model: {model_name}...")
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

print(f"Model loaded: {model_name}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size:,}")
print()

In [ ]:
# Cell 3: Define dataset class
class ConversationDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_length=128):
        self.data = pd.read_csv(csv_path)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.prefix = "Write a reply in your normal texting style:"
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        message = str(self.data.iloc[idx]['message'])
        response = str(self.data.iloc[idx]['response'])
        
        input_text = self.prefix + message
        
        input_encoding = self.tokenizer(
            input_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        target_encoding = self.tokenizer(
            response,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        labels = target_encoding['input_ids'].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_encoding['input_ids'].squeeze(0),
            'attention_mask': input_encoding['attention_mask'].squeeze(0),
            'labels': labels.squeeze(0)
        }

In [ ]:
# Cell 4: Load Your Data
train_csv_path = "../data/processed/mr_train.csv"
val_csv_path = "../data/processed/mr_val.csv"

train_dataset = ConversationDataset(train_csv_path, tokenizer)
val_dataset = ConversationDataset(val_csv_path, tokenizer)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

In [ ]:
# Cell 5: Create DataLoaders
batch_size = 8

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

print(f"Train batches: {len(train_dataloader)}")
print(f"Val batches: {len(val_dataloader)}")

In [ ]:
def train_with_style(
    model,
    tokenizer,
    train_loader,
    val_loader,
    epochs=3,
    lr=3e-4,
    lambda_style=0.1
):
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")

        model.train()
        train_loss = 0

        for batch_idx, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # ---- CE LOSS ----
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            ce_loss = outputs.loss

            # ---- STYLE LOSS (NO GRAD) ----
            with torch.no_grad():
                generated_ids = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_length=64
                )
                generated_texts = tokenizer.batch_decode(
                    generated_ids,
                    skip_special_tokens=True
                )

                style_losses = [
                    style_loss(text) for text in generated_texts
                ]
                style_loss_value = torch.tensor(
                    sum(style_losses) / len(style_losses),
                    device=device
                )

            # ---- TOTAL LOSS ----
            loss = ce_loss + lambda_style * style_loss_value

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            if batch_idx % 10 == 0:
                print(
                    f"Batch {batch_idx}/{len(train_loader)}, "
                    f"CE: {ce_loss.item():.4f}, "
                    f"Style: {style_loss_value.item():.4f}"
                )

        avg_train_loss = train_loss / len(train_loader)

        # ---- Validation (CE only) ----
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                val_loss += outputs.loss.item()

        avg_val_loss = val_loss / len(val_loader)
        print(
            f"Train Loss: {avg_train_loss:.4f}, "
            f"Val Loss: {avg_val_loss:.4f}"
        )

    return model


In [ ]:
print("Starting style-aware training...")

trained_model = train_with_style(
    model,
    tokenizer,
    train_dataloader,
    val_dataloader,
    epochs=3,
    lr=3e-4,
    lambda_style=0.1
)

print("Style-aware training complete!")

In [ ]:
# Cell 8: Generate Multiple Responses Function
def generate_responses(model, tokenizer, message, device, num_responses=3):
    input_text = "Reply: " + message
    input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(device)
    
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_length=64,
            num_beams=5,
            num_return_sequences=num_responses,
            early_stopping=True
        )
        # output_ids = model.generate(
        #     input_ids=input_ids,
        #     max_new_tokens=50,
        #     num_return_sequences=num_responses,
        #     do_sample=True,
        #     temperature=0.3,  # Increased from 0.8 (more randomness)
        #     top_p=0.9,       # Nucleus sampling for diversity
        #     top_k=50,        # Limit to top 50 tokens
        #     repetition_penalty=1.2,  # Lower penalty = more generic
        #     num_beams=5,     # Add beams for better quality
        # )
        
		
    
    responses = []
    for i in range(num_responses):
        response = tokenizer.decode(output_ids[i], skip_special_tokens=True)
        responses.append(response)
    
    return responses

In [ ]:
# Cell 9: Test Model
test_messages = [
    "Hey, what's up?",
    "How are you doing?",
    "See you tomorrow",
    "have you finished the report?"
]

print("Testing model...")
for msg in test_messages:
    response = generate_responses(trained_model, tokenizer, msg, device)
    print(f"Input: {msg}")
    print(f"Response: {response}")
    print("-" * 40)

In [ ]:
# Cell 10: Save Model
save_path = "../models/style_fine_tuned"
trained_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")